# LoRA Contrastive Training for Code-Feedback Alignment

This notebook implements sophisticated contrastive learning to align code snippets with their feedback using:

## Architecture
- **Base Model**: Pre-trained code encoder (e.g., CodeBERT, StarCoder)
- **Fine-tuning**: LoRA (Low-Rank Adaptation) for efficient training
- **Loss**: InfoNCE (contrastive learning)

## Negative Sampling Strategies
1. **Random Negatives**: Random sampling from batch (easy negatives)
2. **Cluster-Based Negatives**: Sample from same cluster for hard negatives

## Custom Metrics
- Positive-negative margin
- Retrieval accuracy (top-1, top-5)
- Mean reciprocal rank (MRR)

## 1. Setup and Imports

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path
import json
from tqdm.auto import tqdm
from typing import Dict, List, Optional, Tuple

# Transformers & PEFT
from transformers import (
    AutoModel,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    get_linear_schedule_with_warmup
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel
)

# Datasets
from datasets import load_dataset, Dataset as HFDataset

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_seed(42)

print("✓ Libraries loaded successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

✓ Libraries loaded successfully!
PyTorch version: 2.9.1
Device: mps


## 2. Configuration

In [21]:
class Config:
    # Model
    model_name = "Salesforce/SFR-Embedding-Code-400M_R"
    
    # LoRA config
    lora_r = 16  # Low-rank dimension
    lora_alpha = 32  # Scaling factor
    lora_dropout = 0.1
    lora_target_modules = ["qkv_proj",      # Attention: Query, Key, Value (fusionnés ici)
        "o_proj",        # Attention: Output projection
        "down_proj",     # MLP: Projection vers le bas
        "up_gate_proj"]  # Attention layers to apply LoRA
    
    # Training
    batch_size = 32  # For in-batch negatives
    learning_rate = 2e-4
    weight_decay = 0.01
    num_epochs = 10
    warmup_steps = 500
    gradient_accumulation_steps = 1
    
    # InfoNCE
    temperature = 0.07  # Temperature for softmax
    
    # Negative sampling
    negative_strategy = "cluster"  # "random" or "cluster"
    
    # Data
    max_code_length = 512
    max_feedback_length = 256
    
    # Paths
    dataset_name = "matis35/RAFT"  # Hugging Face dataset
    clustered_dataset_path = "../data/dataset_clustered_no_tests.csv"
    output_dir = "./checkpoints/lora_contrastive"
    
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else 
                          "cpu" if torch.backends.mps.is_available() else "cpu")
    
config = Config()

print("="*100)
print("CONFIGURATION")
print("="*100)
print(f"Model: {config.model_name}")
print(f"LoRA r={config.lora_r}, alpha={config.lora_alpha}")
print(f"Batch size: {config.batch_size}")
print(f"Learning rate: {config.learning_rate}")
print(f"Temperature: {config.temperature}")
print(f"Negative strategy: {config.negative_strategy}")
print(f"Dataset: {config.dataset_name}")
print(f"Device: {config.device}")

CONFIGURATION
Model: Salesforce/SFR-Embedding-Code-400M_R
LoRA r=16, alpha=32
Batch size: 32
Learning rate: 0.0002
Temperature: 0.07
Negative strategy: cluster
Dataset: matis35/RAFT
Device: cpu


## 3. Load Data with Clustering Information

In [11]:
print("="*100)
print("LOADING DATA")
print("="*100)
print()

# Load dataset from Hugging Face (combine ALL splits)
print("Loading dataset from Hugging Face (matis35/RAFT)...")
hf_dataset = load_dataset("matis35/RAFT")
print(f"  Available splits: {list(hf_dataset.keys())}")

# Combine all splits (train + test) to maximize clustering information
all_dfs = []
for split_name, split_data in hf_dataset.items():
    split_df = split_data.to_pandas()
    print(f"  - {split_name}: {len(split_df):,} samples")
    all_dfs.append(split_df)

df_feedback = pd.concat(all_dfs, ignore_index=True)
print(f"✓ Combined all splits: {len(df_feedback):,} total samples")
print(f"  Columns: {list(df_feedback.columns)}")

# Load clustering information
df_clustered = pd.read_csv(config.clustered_dataset_path)
print(f"✓ Loaded {len(df_clustered):,} samples with clustering info")
print(f"  Columns: {list(df_clustered.columns)}")

df_clustered = df_clustered.rename(columns={'code_snippet': 'code'})

# Merge datasets on code == code
print(f"\nMerging datasets on code == code...")
df = df_feedback.merge(
    df_clustered[['code', 'cluster_kmeans']],
    on='code',
    how='inner'
)

print(f"\n✓ Merged dataset: {len(df):,} samples")
print(f"  Available columns: {list(df.columns)}")

# Check for any missing clusters
missing_clusters = df['cluster_kmeans'].isna().sum()
if missing_clusters > 0:
    print(f"\nWarning: {missing_clusters} samples without cluster info (filling with -1)")
    df['cluster_kmeans'] = df['cluster_kmeans'].fillna(-1).astype(int)
else:
    df['cluster_kmeans'] = df['cluster_kmeans'].astype(int)

print(f"\nCluster distribution:")
print(df['cluster_kmeans'].value_counts().sort_index())

print(f"\n✓ Final columns: {list(df.columns)}")

# NOW create train/val/test splits (80/10/10) using STRATIFICATION on clusters
from sklearn.model_selection import train_test_split

# Check if we can stratify (need enough samples per cluster)
cluster_counts = df['cluster_kmeans'].value_counts()
min_cluster_size = cluster_counts.min()
print(f"\nSmallest cluster has {min_cluster_size} samples")

if min_cluster_size >= 3:
    # Can stratify - this ensures each cluster is represented in train/val/test
    train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['cluster_kmeans'])
    train_df, val_df = train_test_split(train_df, test_size=0.111, random_state=42, stratify=train_df['cluster_kmeans'])
    print("✓ Using stratified splits (each cluster represented in all splits)")
else:
    # Cannot stratify
    train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
    train_df, val_df = train_test_split(train_df, test_size=0.111, random_state=42)
    print("✓ Using random splits (cluster sizes too small for stratification)")

print(f"\nDataset splits:")
print(f"  Train: {len(train_df):,} samples")
print(f"  Val:   {len(val_df):,} samples")
print(f"  Test:  {len(test_df):,} samples")

# Verify cluster distribution in each split
print(f"\nCluster distribution verification:")
print(f"  Train clusters: {train_df['cluster_kmeans'].nunique()}")
print(f"  Val clusters: {val_df['cluster_kmeans'].nunique()}")
print(f"  Test clusters: {test_df['cluster_kmeans'].nunique()}")

LOADING DATA

Loading dataset from Hugging Face (matis35/RAFT)...
  Available splits: ['train', 'validation', 'test']
  - train: 9,445 samples
  - validation: 1,180 samples
  - test: 1,181 samples
✓ Combined all splits: 11,806 total samples
  Columns: ['code', 'feedback']
✓ Loaded 12,232 samples with clustering info
  Columns: ['code_id', 'author_id', 'code_snippet', 'cluster_kmeans', 'tsne_x', 'tsne_y', 'generated_feedback']

Merging datasets on code == code...

✓ Merged dataset: 11,806 samples
  Available columns: ['code', 'feedback', 'cluster_kmeans']

Cluster distribution:
cluster_kmeans
0     876
1    1701
2     632
3    1681
4    1387
5     899
6     836
7     916
8     892
9    1986
Name: count, dtype: int64

✓ Final columns: ['code', 'feedback', 'cluster_kmeans']

Smallest cluster has 632 samples
✓ Using stratified splits (each cluster represented in all splits)

Dataset splits:
  Train: 9,445 samples
  Val:   1,180 samples
  Test:  1,181 samples

Cluster distribution verificat

## 4. Custom Dataset with Negative Sampling

In [12]:
class ContrastiveCodeDataset(Dataset):
    """Dataset for contrastive learning with cluster-aware negative sampling."""
    
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_code_length: int = 512,
        max_feedback_length: int = 256,
        negative_strategy: str = "random"
    ):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_code_length = max_code_length
        self.max_feedback_length = max_feedback_length
        self.negative_strategy = negative_strategy
        
        # Build cluster index for cluster-based sampling
        self.cluster_to_indices = {}
        for idx, cluster in enumerate(df['cluster_kmeans']):
            if cluster not in self.cluster_to_indices:
                self.cluster_to_indices[cluster] = []
            self.cluster_to_indices[cluster].append(idx)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        return {
            'code': row['code'],
            'feedback': row['feedback'],
            'cluster': row['cluster_kmeans'],
            'idx': idx
        }
    
    def get_cluster_batch_indices(self, batch_size: int) -> List[int]:
        """Sample a batch from similar clusters (for hard negatives)."""
        # Randomly select a cluster
        cluster = np.random.choice(list(self.cluster_to_indices.keys()))
        cluster_indices = self.cluster_to_indices[cluster]
        
        # If cluster is too small, mix with random samples
        if len(cluster_indices) < batch_size:
            # Take all from cluster
            sampled = cluster_indices.copy()
            # Fill rest with random
            remaining = batch_size - len(sampled)
            all_indices = list(range(len(self.df)))
            random_indices = np.random.choice(
                [i for i in all_indices if i not in sampled],
                size=remaining,
                replace=False
            )
            sampled.extend(random_indices.tolist())
        else:
            # Sample from cluster
            sampled = np.random.choice(cluster_indices, size=batch_size, replace=False).tolist()
        
        return sampled

print("✓ ContrastiveCodeDataset class defined")

✓ ContrastiveCodeDataset class defined


## 5. Custom Collator for In-Batch Negatives

In [15]:
class ContrastiveCollator:
    """Collator that tokenizes code and feedback separately for contrastive learning."""
    
    def __init__(
        self,
        tokenizer,
        max_code_length: int = 512,
        max_feedback_length: int = 256
    ):
        self.tokenizer = tokenizer
        self.max_code_length = max_code_length
        self.max_feedback_length = max_feedback_length
    
    def __call__(self, batch):
        # Extract codes and feedbacks
        codes = [item['code'] for item in batch]
        feedbacks = [item['feedback'] for item in batch]
        clusters = torch.tensor([item['cluster'] for item in batch])
        indices = torch.tensor([item['idx'] for item in batch])
        
        # Tokenize codes
        code_encodings = self.tokenizer(
            codes,
            padding=True,
            truncation=True,
            max_length=self.max_code_length,
            return_tensors='pt'
        )
        
        # Tokenize feedbacks
        feedback_encodings = self.tokenizer(
            feedbacks,
            padding=True,
            truncation=True,
            max_length=self.max_feedback_length,
            return_tensors='pt'
        )
        
        return {
            'code_input_ids': code_encodings['input_ids'],
            'code_attention_mask': code_encodings['attention_mask'],
            'feedback_input_ids': feedback_encodings['input_ids'],
            'feedback_attention_mask': feedback_encodings['attention_mask'],
            'clusters': clusters,
            'indices': indices
        }

print("✓ ContrastiveCollator class defined")

✓ ContrastiveCollator class defined


## 6. Contrastive Model with LoRA

In [16]:
class ContrastiveCodeModel(nn.Module):
    """Dual encoder model for code-feedback alignment."""
    
    def __init__(
        self,
        base_model_name: str,
        lora_config: LoraConfig,
        temperature: float = 0.07
    ):
        super().__init__()
        
        # Load base model
        self.encoder = AutoModel.from_pretrained(base_model_name)
        
        # Apply LoRA
        self.encoder = get_peft_model(self.encoder, lora_config)
        
        self.temperature = temperature
        
        print(f"\nModel architecture:")
        self.encoder.print_trainable_parameters()
    
    def mean_pooling(self, token_embeddings, attention_mask):
        """Mean pooling with attention mask."""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask
    
    def encode(self, input_ids, attention_mask):
        """Encode input to dense vector."""
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        # L2 normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings
    
    def forward(
        self,
        code_input_ids,
        code_attention_mask,
        feedback_input_ids,
        feedback_attention_mask
    ):
        # Encode codes and feedbacks
        code_embeddings = self.encode(code_input_ids, code_attention_mask)
        feedback_embeddings = self.encode(feedback_input_ids, feedback_attention_mask)
        
        return code_embeddings, feedback_embeddings

print("✓ ContrastiveCodeModel class defined")

✓ ContrastiveCodeModel class defined


## 7. InfoNCE Loss

In [18]:
class InfoNCELoss(nn.Module):
    """InfoNCE loss for contrastive learning with in-batch negatives."""
    
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(
        self,
        code_embeddings: torch.Tensor,
        feedback_embeddings: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Args:
            code_embeddings: (batch_size, embed_dim)
            feedback_embeddings: (batch_size, embed_dim)
        
        Returns:
            loss: scalar
            metrics: dict of metrics
        """
        batch_size = code_embeddings.shape[0]
        
        # Compute similarity matrix: (batch_size, batch_size)
        # similarity[i, j] = cosine_similarity(code_i, feedback_j)
        similarity = torch.matmul(code_embeddings, feedback_embeddings.T) / self.temperature
        
        # Labels: diagonal is positive (code_i matches feedback_i)
        labels = torch.arange(batch_size, device=similarity.device)
        
        # Cross-entropy loss (both directions)
        loss_code_to_feedback = F.cross_entropy(similarity, labels)
        loss_feedback_to_code = F.cross_entropy(similarity.T, labels)
        loss = (loss_code_to_feedback + loss_feedback_to_code) / 2
        
        # Compute metrics
        with torch.no_grad():
            # Positive similarities (diagonal)
            positive_sim = torch.diagonal(similarity).mean()
            
            # Negative similarities (off-diagonal)
            mask = torch.eye(batch_size, device=similarity.device).bool()
            negative_sim = similarity.masked_select(~mask).mean()
            
            # Margin
            margin = positive_sim - negative_sim
            
            # Accuracy (top-1)
            preds = similarity.argmax(dim=1)
            accuracy = (preds == labels).float().mean()
        
        metrics = {
            'positive_sim': positive_sim.item(),
            'negative_sim': negative_sim.item(),
            'margin': margin.item(),
            'accuracy': accuracy.item()
        }
        
        return loss, metrics

print("✓ InfoNCELoss class defined")

✓ InfoNCELoss class defined


## 8. Custom Trainer with Contrastive Learning

In [22]:
class ContrastiveTrainer:
    """Custom trainer for contrastive learning."""
    
    def __init__(
        self,
        model: ContrastiveCodeModel,
        train_dataloader: DataLoader,
        val_dataloader: DataLoader,
        optimizer: torch.optim.Optimizer,
        scheduler,
        loss_fn: InfoNCELoss,
        device: torch.device,
        output_dir: str,
        gradient_accumulation_steps: int = 1
    ):
        self.model = model
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.loss_fn = loss_fn
        self.device = device
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.gradient_accumulation_steps = gradient_accumulation_steps
        
        self.global_step = 0
        self.best_val_loss = float('inf')
        self.history = []
    
    def train_epoch(self, epoch: int):
        """Train for one epoch."""
        self.model.train()
        
        epoch_loss = 0
        epoch_metrics = {
            'positive_sim': 0,
            'negative_sim': 0,
            'margin': 0,
            'accuracy': 0
        }
        
        progress_bar = tqdm(self.train_dataloader, desc=f"Epoch {epoch}")
        
        for step, batch in enumerate(progress_bar):
            # Move to device
            batch = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v 
                    for k, v in batch.items()}
            
            # Forward pass
            code_embeddings, feedback_embeddings = self.model(
                code_input_ids=batch['code_input_ids'],
                code_attention_mask=batch['code_attention_mask'],
                feedback_input_ids=batch['feedback_input_ids'],
                feedback_attention_mask=batch['feedback_attention_mask']
            )
            
            # Compute loss
            loss, metrics = self.loss_fn(code_embeddings, feedback_embeddings)
            loss = loss / self.gradient_accumulation_steps
            
            # Backward pass
            loss.backward()
            
            # Update weights
            if (step + 1) % self.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()
                self.global_step += 1
            
            # Accumulate metrics
            epoch_loss += loss.item() * self.gradient_accumulation_steps
            for k, v in metrics.items():
                epoch_metrics[k] += v
            
            # Update progress bar
            progress_bar.set_postfix({
                'loss': loss.item() * self.gradient_accumulation_steps,
                'margin': metrics['margin'],
                'acc': metrics['accuracy']
            })
        
        # Average metrics
        num_batches = len(self.train_dataloader)
        epoch_loss /= num_batches
        for k in epoch_metrics:
            epoch_metrics[k] /= num_batches
        
        return epoch_loss, epoch_metrics
    
    @torch.no_grad()
    def validate(self):
        """Validate on validation set."""
        self.model.eval()
        
        val_loss = 0
        val_metrics = {
            'positive_sim': 0,
            'negative_sim': 0,
            'margin': 0,
            'accuracy': 0
        }
        
        for batch in tqdm(self.val_dataloader, desc="Validation"):
            # Move to device
            batch = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v 
                    for k, v in batch.items()}
            
            # Forward pass
            code_embeddings, feedback_embeddings = self.model(
                code_input_ids=batch['code_input_ids'],
                code_attention_mask=batch['code_attention_mask'],
                feedback_input_ids=batch['feedback_input_ids'],
                feedback_attention_mask=batch['feedback_attention_mask']
            )
            
            # Compute loss
            loss, metrics = self.loss_fn(code_embeddings, feedback_embeddings)
            
            val_loss += loss.item()
            for k, v in metrics.items():
                val_metrics[k] += v
        
        # Average metrics
        num_batches = len(self.val_dataloader)
        val_loss /= num_batches
        for k in val_metrics:
            val_metrics[k] /= num_batches
        
        return val_loss, val_metrics
    
    def train(self, num_epochs: int):
        """Full training loop."""
        print("="*100)
        print("STARTING TRAINING")
        print("="*100)
        print()
        
        for epoch in range(1, num_epochs + 1):
            # Train
            train_loss, train_metrics = self.train_epoch(epoch)
            
            # Validate
            val_loss, val_metrics = self.validate()
            
            # Log
            print(f"\nEpoch {epoch}/{num_epochs}:")
            print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"  Train Margin: {train_metrics['margin']:.4f} | Val Margin: {val_metrics['margin']:.4f}")
            print(f"  Train Acc: {train_metrics['accuracy']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")
            print(f"  Train Pos/Neg: {train_metrics['positive_sim']:.4f}/{train_metrics['negative_sim']:.4f}")
            print(f"  Val Pos/Neg: {val_metrics['positive_sim']:.4f}/{val_metrics['negative_sim']:.4f}")
            
            # Save history
            self.history.append({
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_metrics': train_metrics,
                'val_metrics': val_metrics
            })
            
            # Save best model
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.save_checkpoint(f"best_model")
                print(f"  ✓ New best model saved (val_loss: {val_loss:.4f})")
            
            # Save checkpoint every epoch
            self.save_checkpoint(f"checkpoint_epoch_{epoch}")
            print()
        
        print("="*100)
        print("TRAINING COMPLETE")
        print("="*100)
    
    def save_checkpoint(self, name: str):
        """Save model checkpoint."""
        checkpoint_dir = self.output_dir / name
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Save LoRA weights
        self.model.encoder.save_pretrained(checkpoint_dir)
        
        # Save training state
        torch.save({
            'global_step': self.global_step,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_val_loss': self.best_val_loss,
            'history': self.history
        }, checkpoint_dir / 'trainer_state.pt')

print("✓ ContrastiveTrainer class defined")

✓ ContrastiveTrainer class defined


## 9. Initialize Model and Tokenizer

In [23]:
print("="*100)
print("INITIALIZING MODEL")
print("="*100)
print()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✓ Tokenizer loaded: {config.model_name}")

# Create LoRA config
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    target_modules=config.lora_target_modules,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)
print(f"✓ LoRA config created")

# Create model
model = ContrastiveCodeModel(
    base_model_name=config.model_name,
    lora_config=lora_config,
    temperature=config.temperature
).to(config.device)
print(f"✓ Model created and moved to {config.device}")

INITIALIZING MODEL

✓ Tokenizer loaded: Salesforce/SFR-Embedding-Code-400M_R
✓ LoRA config created

Model architecture:
trainable params: 7,864,320 || all params: 442,003,456 || trainable%: 1.7792
✓ Model created and moved to cpu


## 10. Create Dataloaders

In [24]:
print("="*100)
print("CREATING DATALOADERS")
print("="*100)
print()

# Create datasets
train_dataset = ContrastiveCodeDataset(
    df=train_df,
    tokenizer=tokenizer,
    max_code_length=config.max_code_length,
    max_feedback_length=config.max_feedback_length,
    negative_strategy=config.negative_strategy
)

val_dataset = ContrastiveCodeDataset(
    df=val_df,
    tokenizer=tokenizer,
    max_code_length=config.max_code_length,
    max_feedback_length=config.max_feedback_length,
    negative_strategy="random"  # Use random for validation
)

print(f"✓ Train dataset: {len(train_dataset):,} samples")
print(f"✓ Val dataset: {len(val_dataset):,} samples")

# Create collator
collator = ContrastiveCollator(
    tokenizer=tokenizer,
    max_code_length=config.max_code_length,
    max_feedback_length=config.max_feedback_length
)

# Create dataloaders
if config.negative_strategy == "cluster":
    # Custom sampler for cluster-based batching
    from torch.utils.data import Sampler
    
    class ClusterBatchSampler(Sampler):
        def __init__(self, dataset, batch_size, shuffle=True):
            self.dataset = dataset
            self.batch_size = batch_size
            self.shuffle = shuffle
        
        def __iter__(self):
            # Generate batches from same cluster
            num_batches = len(self.dataset) // self.batch_size
            for _ in range(num_batches):
                yield self.dataset.get_cluster_batch_indices(self.batch_size)
        
        def __len__(self):
            return len(self.dataset) // self.batch_size
    
    train_sampler = ClusterBatchSampler(train_dataset, config.batch_size)
    train_dataloader = DataLoader(
        train_dataset,
        batch_sampler=train_sampler,
        collate_fn=collator,
        num_workers=0
    )
else:
    # Random sampling
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collator,
        num_workers=0
    )

val_dataloader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

print(f"✓ Train dataloader: {len(train_dataloader)} batches")
print(f"✓ Val dataloader: {len(val_dataloader)} batches")
print(f"\nNegative sampling strategy: {config.negative_strategy}")

CREATING DATALOADERS

✓ Train dataset: 9,445 samples
✓ Val dataset: 1,180 samples
✓ Train dataloader: 295 batches
✓ Val dataloader: 37 batches

Negative sampling strategy: cluster


## 11. Setup Training

In [25]:
print("="*100)
print("SETUP TRAINING")
print("="*100)
print()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)
print(f"✓ Optimizer: AdamW (lr={config.learning_rate})")

# Scheduler
num_training_steps = len(train_dataloader) * config.num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=config.warmup_steps,
    num_training_steps=num_training_steps
)
print(f"✓ Scheduler: Linear warmup ({config.warmup_steps} steps) + decay")

# Loss function
loss_fn = InfoNCELoss(temperature=config.temperature)
print(f"✓ Loss: InfoNCE (temperature={config.temperature})")

# Trainer
trainer = ContrastiveTrainer(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=loss_fn,
    device=config.device,
    output_dir=config.output_dir,
    gradient_accumulation_steps=config.gradient_accumulation_steps
)
print(f"✓ Trainer initialized")
print(f"\nTotal training steps: {num_training_steps:,}")

SETUP TRAINING

✓ Optimizer: AdamW (lr=0.0002)
✓ Scheduler: Linear warmup (500 steps) + decay
✓ Loss: InfoNCE (temperature=0.07)
✓ Trainer initialized

Total training steps: 2,950


## 12. Train Model

In [26]:
# Start training
trainer.train(num_epochs=config.num_epochs)

STARTING TRAINING



Epoch 1:   0%|          | 0/295 [00:00<?, ?it/s]

: 

## 13. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

history = trainer.history
epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
train_margins = [h['train_metrics']['margin'] for h in history]
val_margins = [h['val_metrics']['margin'] for h in history]
train_accs = [h['train_metrics']['accuracy'] for h in history]
val_accs = [h['val_metrics']['accuracy'] for h in history]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(epochs, train_losses, 'b-', label='Train')
axes[0, 0].plot(epochs, val_losses, 'r-', label='Validation')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Margin
axes[0, 1].plot(epochs, train_margins, 'b-', label='Train')
axes[0, 1].plot(epochs, val_margins, 'r-', label='Validation')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Margin (Pos - Neg)')
axes[0, 1].set_title('Positive-Negative Similarity Margin')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Accuracy
axes[1, 0].plot(epochs, train_accs, 'b-', label='Train')
axes[1, 0].plot(epochs, val_accs, 'r-', label='Validation')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Retrieval Accuracy (Top-1)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Positive vs Negative similarity
train_pos = [h['train_metrics']['positive_sim'] for h in history]
train_neg = [h['train_metrics']['negative_sim'] for h in history]
axes[1, 1].plot(epochs, train_pos, 'g-', label='Positive (Train)')
axes[1, 1].plot(epochs, train_neg, 'orange', label='Negative (Train)')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Similarity')
axes[1, 1].set_title('Positive vs Negative Similarity')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.output_dir / 'training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Training history saved to: {config.output_dir / 'training_history.png'}")

## 14. Evaluate on Test Set

In [ ]:
print("="*100)
print("EVALUATING ON TEST SET")
print("="*100)
print()

# Load best model
best_model_path = config.output_dir / "best_model"
model.encoder = PeftModel.from_pretrained(model.encoder.base_model, best_model_path)
print(f"✓ Loaded best model from: {best_model_path}")

# Create test dataloader
test_dataset = ContrastiveCodeDataset(
    df=test_df,
    tokenizer=tokenizer,
    max_code_length=config.max_code_length,
    max_feedback_length=config.max_feedback_length,
    negative_strategy="random"
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

print(f"✓ Test dataset: {len(test_dataset):,} samples")

# Evaluate
model.eval()
test_loss = 0
test_metrics = {
    'positive_sim': 0,
    'negative_sim': 0,
    'margin': 0,
    'accuracy': 0
}

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Testing"):
        batch = {k: v.to(config.device) if isinstance(v, torch.Tensor) else v 
                for k, v in batch.items()}
        
        code_embeddings, feedback_embeddings = model(
            code_input_ids=batch['code_input_ids'],
            code_attention_mask=batch['code_attention_mask'],
            feedback_input_ids=batch['feedback_input_ids'],
            feedback_attention_mask=batch['feedback_attention_mask']
        )
        
        loss, metrics = loss_fn(code_embeddings, feedback_embeddings)
        
        test_loss += loss.item()
        for k, v in metrics.items():
            test_metrics[k] += v

num_batches = len(test_dataloader)
test_loss /= num_batches
for k in test_metrics:
    test_metrics[k] /= num_batches

print("\n" + "="*100)
print("TEST RESULTS")
print("="*100)
print(f"Loss: {test_loss:.4f}")
print(f"Margin (Pos - Neg): {test_metrics['margin']:.4f}")
print(f"Accuracy (Top-1): {test_metrics['accuracy']:.4f}")
print(f"Positive Similarity: {test_metrics['positive_sim']:.4f}")
print(f"Negative Similarity: {test_metrics['negative_sim']:.4f}")
print("="*100)

## 15. Save Training Report

In [ ]:
# Create training report
report = f"""# LoRA Contrastive Training Report

## Configuration

- **Model**: {config.model_name}
- **LoRA**: r={config.lora_r}, alpha={config.lora_alpha}, dropout={config.lora_dropout}
- **Loss**: InfoNCE (temperature={config.temperature})
- **Batch size**: {config.batch_size}
- **Learning rate**: {config.learning_rate}
- **Epochs**: {config.num_epochs}
- **Negative sampling**: {config.negative_strategy}
- **Device**: {config.device}

## Dataset

- Train: {len(train_df):,} samples
- Validation: {len(val_df):,} samples
- Test: {len(test_df):,} samples

## Training Results

### Best Validation Performance

- **Best Epoch**: {history[np.argmin([h['val_loss'] for h in history])]['epoch']}
- **Best Val Loss**: {trainer.best_val_loss:.4f}
- **Best Val Margin**: {history[np.argmin([h['val_loss'] for h in history])]['val_metrics']['margin']:.4f}
- **Best Val Accuracy**: {history[np.argmin([h['val_loss'] for h in history])]['val_metrics']['accuracy']:.4f}

### Test Performance

- **Test Loss**: {test_loss:.4f}
- **Test Margin**: {test_metrics['margin']:.4f}
- **Test Accuracy**: {test_metrics['accuracy']:.4f}
- **Positive Similarity**: {test_metrics['positive_sim']:.4f}
- **Negative Similarity**: {test_metrics['negative_sim']:.4f}

## Model Checkpoint

Best model saved to: `{config.output_dir / 'best_model'}`

## Usage

```python
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel

# Load base model and tokenizer
base_model = AutoModel.from_pretrained("{config.model_name}")
tokenizer = AutoTokenizer.from_pretrained("{config.model_name}")

# Load LoRA weights
model = PeftModel.from_pretrained(base_model, "{config.output_dir / 'best_model'}")

# Encode code
code = "int factorial(int n) {{ return n == 0 ? 1 : n * factorial(n-1); }}"
inputs = tokenizer(code, return_tensors="pt")
outputs = model(**inputs)
embedding = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
```
"""

# Save report
report_path = config.output_dir / 'training_report.md'
with open(report_path, 'w') as f:
    f.write(report)

print(f"✓ Training report saved to: {report_path}")
print()
print(report)

## Summary

This notebook has:

1. ✓ Implemented dual encoder with LoRA for code-feedback alignment
2. ✓ Used InfoNCE loss with in-batch negatives
3. ✓ Implemented two negative sampling strategies:
   - Random negatives (easy)
   - Cluster-based negatives (hard)
4. ✓ Tracked custom metrics:
   - Positive-negative margin
   - Retrieval accuracy
   - Similarity distributions
5. ✓ Trained and evaluated the model
6. ✓ Saved best checkpoint

**Next steps**:
- Compare random vs cluster-based negative sampling
- Experiment with different temperatures
- Try different base models (CodeT5, StarCoder, etc.)
- Implement retrieval evaluation (MRR, NDCG)